# SD3.5 CONCEPT LoRA — all-in-one (train → generate → contact sheet)

Plain **text→image** LoRA: the model learns to GENERATE images like the PIPE
`add a person` subset. There is **no** mask, **no** source image, **no** input_proj,
**no** segment/paste — it is a straight DreamBooth-style concept LoRA. Only the
finished photo (`target_img`) is used; captions are derived from the PIPE
instruction (imperative stripped).

Pipeline: setup → SD3.5 access → train (ONE artifact: LoRA, zip immediately) →
generate in a fresh subprocess → contact sheet (one tile per validation prompt).

Trains ONE artifact (`pytorch_lora_weights.safetensors`). To run unattended set
`SMOKE = False` then **Save Version → Save & Run All**. Needs GPU + SD3.5 access
(HF_TOKEN secret or a mounted model dataset).

## 1. Setup — install pinned stack (batch-safe, no kernel restart)

In [ ]:
import subprocess, sys, os
from pathlib import Path
REPO = Path('/kaggle/working/VIN')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/BDT-17/VIN.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin'], check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'], check=True)
sys.path.insert(0, str(REPO))
print('repo at', subprocess.run(['git','-C',str(REPO),'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip())
subprocess.run([sys.executable,'-m','pip','install','-q','--force-reinstall','--no-deps',
                'transformers==4.46.3','tokenizers==0.20.3','huggingface_hub==0.25.2'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q',
                'diffusers==0.31.0','accelerate==0.34.2','peft==0.13.2','datasets>=2.20',
                'safetensors>=0.4.3','sentencepiece','protobuf','pillow>=10','numpy'], check=True)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import transformers, diffusers, torch
print('transformers', transformers.__version__, '| diffusers', diffusers.__version__)
assert transformers.__version__ == '4.46.3', f'transformers is {transformers.__version__}, expected 4.46.3'
from transformers.utils import FLAX_WEIGHTS_NAME
assert torch.cuda.is_available(), 'No GPU — set Accelerator to GPU'
print('OK on', torch.cuda.get_device_name(0))

## 2. SD3.5 access (gated)

In [ ]:
from pathlib import Path
_local = Path('/kaggle/input/stable-diffusion-3-5-medium')
HF_TOKEN = None
if _local.exists():
    SD35_MODEL = str(_local); print('local SD3.5 mount:', SD35_MODEL)
else:
    SD35_MODEL = 'stabilityai/stable-diffusion-3.5-medium'
    try:
        from kaggle_secrets import UserSecretsClient; HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        import os; HF_TOKEN = os.environ.get('HF_TOKEN')
    assert HF_TOKEN, 'SD3.5 is gated. Add a Kaggle secret HF_TOKEN or mount the model dataset.'
    from huggingface_hub import login; login(token=HF_TOKEN); print('HF login OK')
WORK = Path('/kaggle/working/vin_lora')

## 3. Train the concept LoRA on PIPE person images

Smoke first (200 steps / 200 images). For a real adapter set `SMOKE = False`
(config: 4000 steps / 4000 images, grad-accum 4, lr 1e-4).

In [ ]:
from LoRA.train.train_concept_lora import run_training
import shutil
SMOKE = True   # True = 200 steps/200 images (fast); False = full (4000/4000)
kw = dict(max_train_steps=200, num_train_samples=200) if SMOKE else {}
train = run_training(WORK, base_model_id=SD35_MODEL, hf_token=HF_TOKEN, **kw)
RUN_DIR = train['run_dir']
print('trained ->', RUN_DIR)
ZIP = shutil.make_archive(str(RUN_DIR), 'zip', str(RUN_DIR))
print('adapter zip ->', ZIP, '(download this / make it a dataset)')
prov = train['provenance']

# Free the training pipeline's VRAM BEFORE the generate subprocess. The trainer
# leaves a few GB resident on this kernel's GPU; the subprocess starts a second
# process and OOMs loading its own pipeline if we don't release this first.
import gc, torch
del train
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache(); torch.cuda.synchronize()
    free_mb = (torch.cuda.get_device_properties(0).total_memory
               - torch.cuda.memory_reserved(0)) / 1024**2
    print(f'VRAM free after train cleanup: {free_mb:.0f} MB (want most of ~15000)')
prov

## 4. Generate in a FRESH process (avoids train+reload RAM OOM)

Generates one image per `validation_prompts` entry from the run's training_config.

In [ ]:
import subprocess, sys
cmd = [sys.executable, '-m', 'LoRA.inference.run_concept_eval',
       '--run-dir', str(RUN_DIR), '--base-model', str(SD35_MODEL),
       '--steps', '28', '--guidance', '7.0']
if HF_TOKEN:
    cmd += ['--hf-token', HF_TOKEN]
print('running generate subprocess...', flush=True)
# capture so the subprocess traceback is visible in the notebook (not swallowed)
r = subprocess.run(cmd, cwd='/kaggle/working/VIN', capture_output=True, text=True)
print('eval exit code:', r.returncode)
print('--- STDOUT (tail) ---'); print(r.stdout[-2500:])
if r.returncode != 0:
    print('--- STDERR (tail) ---'); print(r.stderr[-5000:])
EVAL_OUT = RUN_DIR / 'concept_eval'

## 5. Show the contact sheet (one tile per validation prompt)

In [ ]:
from pathlib import Path
sheet = EVAL_OUT / 'contact_sheet.png'
assert sheet.exists(), f'generate did not produce a contact sheet (check subprocess output above): {sheet}'
print('contact sheet:', sheet)
from IPython.display import Image as IPImage, display
display(IPImage(str(sheet)))